<a href="https://colab.research.google.com/github/Lau-Tisca/FlyRank_ML_1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lau-Tisca/FlyRank_ML_1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Unit of Analysis (Grain):** One row represents one unique pseudonymized content item (page) over our aggregated analysis window.
* **Target Lane:** Lane 2 — Refresh / Content Opportunity Scoring.
* **Primary Tables Used:**
    * `dim_content` (for static content metadata).
    * `fact_content_daily_performance` (restricted to partition `month=2026-03` for testing, verification, and feature engineering).
* **Time Windows:**
    * **Feature Window:** March 1, 2026, to March 15, 2026 (15 days).
    * **Label / Target Window:** March 16, 2026, to March 31, 2026 (16 days).
* **The Target (Proxy Label):** Traffic decline status, defined as:
    $$\text{is_declining} = (\text{clicks_future_16d} < 0.8 \times \text{clicks_prior_15d})$$
    Where clicks decline by more than 20% in the second half of March compared to the first half.
* **Deliberately Excluded Fields:**
    * `client_hash_id` and `content_hash_id` as model features (used strictly for joins, grouping, and client-holdout splits to avoid memorization).
    * `trend_direction` and `trend_pct` (from raw sources) because they are directly derived from target-period metrics, causing absolute leakage.

In [ ]:
import os
import getpass
import duckdb

# Fetch Token securely (checks Env variables first, then Google Secrets, then prompts)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

# Connect DuckDB and register HF credentials
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Configure dataset paths
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':      f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':      f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_march': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_query_90d':   f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected and tables configured successfully!")

DuckDB connected and tables configured successfully!


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Column Name | Classification | Source Table | Reason / Role |
| :--- | :--- | :--- | :--- |
| `content_hash_id` | Context | `dim_content` / `fact_daily` | Grouping, joining, and train/test deduplication only. |
| `client_hash_id` | Context | `dim_content` / `fact_daily` | Grouped train/test splitting (client-holdout validation). |
| `gsc_impressions` | Feature (aggregated) | `fact_daily` | Measures past search visibility and organic demand. |
| `gsc_clicks` | Feature (aggregated) | `fact_daily` | Measures baseline organic search traffic. |
| `gsc_avg_position` | Feature (averaged) | `fact_daily` | Measures baseline search engine ranking visibility. |
| `ga4_sessions` | Feature (aggregated) | `fact_daily` | Measures baseline on-site organic session traffic. |
| `is_declining` | Label / Target | Derived | The true prediction outcome (decline > 20% in the future window). |
| `clicks_leak` | Excluded / Leakage | Derived | Derived from the future window (March 16-31); excluded to prevent target leakage. |
| `trend_direction` | Excluded | Derived | Calculated from target-period metrics; excluded to avoid circular modeling. |

In [ ]:
# Inspect columns of the daily fact table to confirm schema and types match our mappings
schema_df = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily_march']}").df()
print("Verified columns in fact_content_daily_performance:")
print(schema_df[['column_name', 'column_type']].to_string(index=False))

Verified columns in fact_content_daily_performance:
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemin

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

We will now run exactly three verification queries to prove our data grains, counts, and GA4 availability:
1.  **Grain Verification:** Proves that grouping by `report_date`, `client_hash_id`, and `content_hash_id` yields exactly one row (zero rows returned on `COUNT(*) > 1` holds the grain).
2.  **Slice Statistics:** Verifies the exact row counts and calendar date spans for the `2026-03` partition.
3.  **GA4 Availability Verification:** Filters with `IS TRUE` to check active GA4 tracking density.

#### Feature Generation Specifications (5 Features Max):
1.  `imp_prior`: Total GSC impressions (March 1st–15th). **Knowable at decision moment** because impressions are fully recorded in Search Console before the prediction window begins.
2.  `clicks_prior`: Total GSC clicks (March 1st–15th). **Knowable at decision moment** because historical organic clicks are settled prior to March 15.
3.  `avg_pos_prior`: Average GSC SERP position (March 1st–15th). **Knowable at decision moment** because position history is measured and frozen before the target window starts.
4.  `sessions_prior`: Total GA4 Sessions (March 1st–15th). **Knowable at decision moment** because analytics sessions are compiled and updated in real-time during the first half of the month.
5.  `ctr_prior`: Organic Search CTR (March 1st–15th). **Knowable at decision moment** because it is mathematically computed from past clicks and impressions.

#### The Leakage Trap Experiment:
We deliberately extract `clicks_leak` (clicks from the future target window March 16th–31st) and feed it into a model to demonstrate validation leakage, before dropping it to retain a mathematically honest scoring baseline.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

# ==========================================
# 1. RUN THE THREE VERIFICATION QUERIES
# ==========================================

print("--- QUERY 1: GRAIN VERIFICATION (Should return empty) ---")
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
    FROM {TABLES['fact_daily_march']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(grain_check)

print("\n--- QUERY 2: SLICE ROW COUNT & DATE SPAN ---")
slice_stats = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES['fact_daily_march']}
""").df()
print(slice_stats.to_string(index=False))

print("\n--- QUERY 3: GA4 AVAILABILITY (IS TRUE Filter) ---")
ga4_stats = con.sql(f"""
    SELECT
        COUNT(*) AS ga4_active_rows,
        ROUND(AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) * 100, 2) AS active_pct
    FROM {TABLES['fact_daily_march']}
    WHERE ga4_data_available IS TRUE
""").df()
print(ga4_stats.to_string(index=False))

# ==========================================
# 2. FEATURE GENERATION & LEAKAGE TRAP
# ==========================================

print("\n--- BUILDING FEATURE FRAME & TARGET WINDOW ---")
feature_query = con.sql(f"""
    WITH pre_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            -- 15-day prior feature window
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_prior,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_prior,
            AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS avg_pos_prior,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN ga4_sessions ELSE 0 END) AS sessions_prior,

            -- 16-day future label window
            SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_future
        FROM {TABLES['fact_daily_march']}
        GROUP BY 1, 2
        HAVING imp_prior >= 50 -- Ensure a reasonable volume floor
    )
    SELECT
        client_hash_id,
        content_hash_id,
        imp_prior,
        clicks_prior,
        COALESCE(avg_pos_prior, 20.0) AS avg_pos_prior,
        sessions_prior,
        ROUND((clicks_prior / NULLIF(imp_prior, 0)) * 100, 2) AS ctr_prior,

        -- Leakage Trap (derived from target window)
        clicks_future AS clicks_leak,

        -- Target Proxy Label
        CASE WHEN clicks_future < 0.8 * clicks_prior THEN 1 ELSE 0 END AS is_declining
    FROM pre_agg
""").df()

print(f"Feature frame successfully extracted: {len(feature_query):,} rows.")

# Prepare Model Inputs
df_clean = feature_query.dropna()
honest_cols = ['imp_prior', 'clicks_prior', 'avg_pos_prior', 'sessions_prior', 'ctr_prior']
leaked_cols = honest_cols + ['clicks_leak']

X_honest = df_clean[honest_cols]
X_leaked = df_clean[leaked_cols]
y = df_clean['is_declining']

# Stratified Split
X_tr_h, X_te_h, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.3, random_state=42, stratify=y)

print("\n--- EXPERIMENT A: WITH LEAKAGE (Clicks_leak included) ---")
model_leak = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1).fit(X_tr_l, y_tr)
y_pred_l = model_leak.predict_proba(X_te_l)[:, 1]
print(f"ROC AUC (Leaked): {roc_auc_score(y_te, y_pred_l):.4f}")
print(f"Accuracy (Leaked): {accuracy_score(y_te, model_leak.predict(X_te_l)):.4f}")

print("\n--- EXPERIMENT B: HONEST MODEL (No Leakage) ---")
model_honest = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1).fit(X_tr_h, y_tr)
y_pred_h = model_honest.predict_proba(X_te_h)[:, 1]
print(f"ROC AUC (Honest): {roc_auc_score(y_te, y_pred_h):.4f}")
print(f"Accuracy (Honest): {accuracy_score(y_te, model_honest.predict(X_te_h)):.4f}")

--- QUERY 1: GRAIN VERIFICATION (Should return empty) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []

--- QUERY 2: SLICE ROW COUNT & DATE SPAN ---
 total_rows   min_date   max_date
    9841378 2026-03-01 2026-03-31

--- QUERY 3: GA4 AVAILABILITY (IS TRUE Filter) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 ga4_active_rows  active_pct
          413966       100.0

--- BUILDING FEATURE FRAME & TARGET WINDOW ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame successfully extracted: 92,548 rows.

--- EXPERIMENT A: WITH LEAKAGE (Clicks_leak included) ---
ROC AUC (Leaked): 0.9999
Accuracy (Leaked): 0.9970

--- EXPERIMENT B: HONEST MODEL (No Leakage) ---
ROC AUC (Honest): 0.8542
Accuracy (Honest): 0.7711


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Unbalanced History & GA4 Initialization:** The dataset is structured as an unbalanced panel. Several clients lack full tracking history at the beginning of the time span, or have `ga4_data_available = FALSE`. A global calendar-based window will systematically mistake uninitialized tracking for zero organic sessions unless filtered explicitly with `ga4_data_available IS TRUE`.
* **Query-Mix Trailing Windows:** The `fact_content_query_90d` table holds a fixed 90-day static window. Using search metrics from this table as model features while predicting a label within that same 90-day time window introduces future overlap leakage.

In [ ]:
# Prove the unbalanced panel limitation by inspecting client tracking starts
unbalanced_panel = con.sql(f"""
    SELECT
        client_hash_id,
        gsc_data_start,
        ga4_data_start,
        DATEDIFF('day', gsc_data_start, DATE '2026-06-30') as days_of_history
    FROM {TABLES['dim_clients']}
    WHERE gsc_data_start IS NOT NULL
    ORDER BY gsc_data_start ASC
    LIMIT 10
""").df()

print("Observed scale and history duration variability across clients:")
print(unbalanced_panel.to_string(index=False))

Observed scale and history duration variability across clients:
         client_hash_id gsc_data_start ga4_data_start  days_of_history
client_9958f0a7ae1df715     2025-01-27     2025-10-29              519
client_ff644d8251367cbb     2025-01-27     2025-10-29              519
client_73cda7b4e4f265ea     2025-02-11     2026-03-24              504
client_fef1a8f436438636     2025-03-11     2026-03-06              476
client_62f4a7e64f5e0096     2025-06-07            NaT              388
client_b10cb2997d0c7c86     2025-06-18     2025-11-15              377
client_65de48885f4ef01b     2025-06-21     2026-02-19              374
client_c182d11e4862a37d     2025-06-21     2026-02-20              374
client_3197e6291363b4db     2025-06-29     2025-11-09              366
client_625b6439094e23e4     2025-07-01     2026-02-19              364


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.